# Experiment Hygiene: Seeds, Checkpoints, and Logging

Good experiment hygiene separates results you can trust from results you can't reproduce. This notebook covers the three habits that matter most: setting random seeds so your runs are deterministic, saving checkpoints so you can resume or recover from crashes, and logging metrics so you can actually understand what happened during training.

In [ ]:
# pip install torch numpy wandb  # uncomment if needed
import random
import os
import csv
import logging
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Random Seeds

Many parts of a training run involve randomness:
- Weight initialization
- Data shuffling / mini-batch ordering
- Data augmentation
- Dropout masks

If you don't fix the seed, every run produces different results. That makes it impossible to distinguish "this change helped" from "this run got lucky."

There are **four** random number generators you need to seed:

| Library | Why it matters |
|---|---|
| `random` | Python's built-in RNG, used by some data loaders and augmentation code |
| `numpy` | Used by NumPy-based augmentation, scikit-learn, and many preprocessing steps |
| `torch` | CPU-side PyTorch ops |
| `torch.cuda` | GPU-side PyTorch ops (separate from CPU) |

Missing even one of these can leave a source of non-determinism in your pipeline.

In [ ]:
def set_seed(seed: int) -> None:
    """Seed all relevant random number generators for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU

    # These two flags make CUDA ops deterministic at a small speed cost.
    # CUBLAS_WORKSPACE_CONFIG must be set before PyTorch uses cuBLAS.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

set_seed(42)
print("Seeds set.")

### A Caveat on `cudnn.benchmark`

Setting `torch.backends.cudnn.benchmark = True` tells PyTorch to benchmark different cuDNN algorithms and pick the fastest one for your specific input shapes. This can speed up training by 10-30%, but the chosen algorithm may vary run-to-run, breaking determinism.

For experiments where you're comparing two changes: set `benchmark = False` and `deterministic = True`. For production training where speed matters more than perfect reproducibility: `benchmark = True` is fine.

### Verifying Seeds Work

In [ ]:
def sample_random_values(seed):
    set_seed(seed)
    py_val = random.random()
    np_val = np.random.rand()
    torch_val = torch.rand(1).item()
    return py_val, np_val, torch_val

run1 = sample_random_values(42)
run2 = sample_random_values(42)
run3 = sample_random_values(99)  # Different seed

print(f"Run with seed=42 (first call):  {run1}")
print(f"Run with seed=42 (second call): {run2}")
print(f"Run with seed=99:               {run3}")
print()
assert run1 == run2, "Same seed should produce identical values"
assert run1 != run3, "Different seeds should produce different values"
print("Same seed -> same values. Different seed -> different values.")

## 2. Saving and Loading Checkpoints

A checkpoint is a snapshot of your training state at a given point. You need them for two reasons:

1. **Recovery**: GPU jobs crash. Servers restart. If you're 8 hours into a 10-hour training run and it crashes, a checkpoint from hour 6 means you only lose 2 hours.
2. **Best-model selection**: Your final epoch isn't always your best epoch. Save checkpoints periodically and reload the one with the lowest validation loss.

### What to Save

A complete checkpoint should contain:

| Key | What it is |
|---|---|
| `model_state_dict` | Learned weights |
| `optimizer_state_dict` | Optimizer momentum buffers, learning rate schedule state |
| `epoch` | Which epoch this was saved at |
| `loss` | The loss value at this checkpoint |
| `config` | Hyperparameters (so you know what produced this checkpoint) |

Saving only the model weights is a common mistake. Without the optimizer state, you can't resume training cleanly because the momentum terms restart from zero.

In [ ]:
def save_checkpoint(
    checkpoint_dir: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    config: dict,
    is_best: bool = False,
) -> str:
    """Save a training checkpoint.
    
    Returns the path the checkpoint was saved to.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    state = {
        "epoch": epoch,
        "loss": loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
    }
    
    path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch:04d}.pt")
    torch.save(state, path)
    
    if is_best:
        best_path = os.path.join(checkpoint_dir, "best.pt")
        torch.save(state, best_path)
    
    return path


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer = None,
    device: torch.device = torch.device("cpu"),
) -> dict:
    """Load a checkpoint. Returns the checkpoint dict (contains epoch, loss, config)."""
    state = torch.load(path, map_location=device)
    model.load_state_dict(state["model_state_dict"])
    
    if optimizer is not None and "optimizer_state_dict" in state:
        optimizer.load_state_dict(state["optimizer_state_dict"])
    
    return state


print("Checkpoint functions defined.")

## 3. Logging Metrics

There are two levels of logging worth setting up:

1. **Console/file logs**: Python's built-in `logging` module. Captures events, errors, and training progress as text. Useful for debugging.
2. **Metric logs**: A structured record of numbers over time (loss, accuracy per epoch). A CSV file works fine; cloud tools like W&B or MLflow are better if you're running many experiments.

### Setting Up a Logger

In [ ]:
def get_logger(name: str, log_file: str = None) -> logging.Logger:
    """Create a logger that writes to console and optionally to a file."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    # Avoid adding duplicate handlers if the logger already exists
    if logger.handlers:
        return logger
    
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    
    # Console handler
    ch = logging.StreamHandler()
    ch.setFormatter(formatter)
    logger.addHandler(ch)
    
    # File handler (optional)
    if log_file:
        os.makedirs(os.path.dirname(log_file) or ".", exist_ok=True)
        fh = logging.FileHandler(log_file)
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    
    return logger

logger = get_logger("train", log_file="runs/demo/train.log")
logger.info("Logger initialized.")

### CSV Metric Logger

For structured metrics, a CSV file is simple, portable, and easy to plot with pandas later.

In [ ]:
class CSVLogger:
    """Logs metrics to a CSV file, one row per epoch."""

    def __init__(self, path: str, fieldnames: list):
        self.path = path
        self.fieldnames = fieldnames
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        
        # Write header
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    def log(self, row: dict):
        """Append a row of metrics. Missing fields are written as empty."""
        with open(self.path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writerow({k: row.get(k, "") for k in self.fieldnames})

    def read(self):
        """Return all logged rows as a list of dicts."""
        with open(self.path, "r") as f:
            return list(csv.DictReader(f))


# Quick demo
csv_logger = CSVLogger(
    path="runs/demo/metrics.csv",
    fieldnames=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
)
csv_logger.log({"epoch": 1, "train_loss": 0.654, "train_acc": 0.612, "val_loss": 0.680, "val_acc": 0.590})
csv_logger.log({"epoch": 2, "train_loss": 0.521, "train_acc": 0.723, "val_loss": 0.545, "val_acc": 0.710})
print(csv_logger.read())

### Weights & Biases (W&B)

W&B is a free cloud tool that stores metrics, hyperparameters, and artifacts (model files). The API is straightforward.

The minimal pattern:

In [ ]:
# This cell shows the W&B pattern without actually running it.
# To use it: pip install wandb, then wandb login in your terminal.

WANDB_AVAILABLE = False
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    print("wandb not installed. Skipping W&B logging.")


def setup_wandb(config: dict, project: str = "til-ai-demo"):
    """Initialize a W&B run. Call once at the start of training."""
    if not WANDB_AVAILABLE:
        return None
    run = wandb.init(project=project, config=config)
    return run


def log_to_wandb(metrics: dict):
    """Log a dict of metrics. Call once per epoch."""
    if not WANDB_AVAILABLE:
        return
    wandb.log(metrics)


# Example usage pattern (not executed here):
example_usage = """
config = {"lr": 1e-3, "epochs": 50, "batch_size": 32, "seed": 42}
run = setup_wandb(config, project="my-experiment")

for epoch in range(config["epochs"]):
    # ... training ...
    log_to_wandb({"epoch": epoch, "train_loss": loss, "val_acc": acc})

wandb.finish()
"""
print("W&B pattern shown above. Logs appear at wandb.ai in your project dashboard.")

## 4. Hands-On: A Complete Training Loop with Hygiene

Now we put it all together: a 2-layer MLP trained on synthetic data, with:
- Seeds set before everything
- Checkpoints saved every 10 epochs (and a `best.pt` whenever validation loss improves)
- Metrics logged to a CSV
- A resume-from-checkpoint section at the end

First, define the model and data.

In [ ]:
set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "seed": 42,
    "lr": 1e-3,
    "epochs": 50,
    "batch_size": 64,
    "hidden_dim": 128,
    "input_dim": 20,
    "checkpoint_every": 10,
    "checkpoint_dir": "runs/mlp_demo/checkpoints",
    "metrics_csv": "runs/mlp_demo/metrics.csv",
}

# --- Synthetic regression problem ---
# Features are 20-dimensional. Target is a noisy linear combination of the first 5.
N_TRAIN, N_VAL = 2000, 400
X_all = torch.randn(N_TRAIN + N_VAL, CONFIG["input_dim"])
true_weights = torch.zeros(CONFIG["input_dim"])
true_weights[:5] = torch.tensor([1.0, -2.0, 0.5, 1.5, -1.0])
y_all = X_all @ true_weights + 0.2 * torch.randn(N_TRAIN + N_VAL)

X_train, y_train = X_all[:N_TRAIN].to(DEVICE), y_all[:N_TRAIN].to(DEVICE)
X_val,   y_val   = X_all[N_TRAIN:].to(DEVICE), y_all[N_TRAIN:].to(DEVICE)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Device: {DEVICE}")

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


model = MLP(CONFIG["input_dim"], CONFIG["hidden_dim"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
criterion = nn.MSELoss()

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)

# Set up logging
run_logger = get_logger("mlp_demo", "runs/mlp_demo/train.log")
run_csv = CSVLogger(
    CONFIG["metrics_csv"],
    fieldnames=["epoch", "train_loss", "val_loss"]
)

best_val_loss = float("inf")

for epoch in range(1, CONFIG["epochs"] + 1):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_loss = epoch_loss / N_TRAIN

    # --- Validation ---
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val)
        val_loss = criterion(val_preds, y_val).item()

    # --- Log ---
    run_csv.log({"epoch": epoch, "train_loss": f"{train_loss:.6f}", "val_loss": f"{val_loss:.6f}"})
    if epoch % 10 == 0:
        run_logger.info(f"Epoch {epoch:3d}/{CONFIG['epochs']} | train={train_loss:.4f} | val={val_loss:.4f}")

    # --- Checkpoint ---
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss

    if epoch % CONFIG["checkpoint_every"] == 0 or is_best:
        ckpt_path = save_checkpoint(
            CONFIG["checkpoint_dir"],
            model, optimizer, epoch, val_loss, CONFIG,
            is_best=is_best,
        )
        if epoch % CONFIG["checkpoint_every"] == 0:
            run_logger.info(f"  Saved checkpoint: {ckpt_path}")
        if is_best:
            run_logger.info(f"  New best val_loss={val_loss:.4f}. Saved best.pt.")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

In [ ]:
# See what was saved
ckpt_dir = Path(CONFIG["checkpoint_dir"])
for f in sorted(ckpt_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"{f.name:40s}  {size_kb:.1f} KB")

### Resuming from a Checkpoint

Suppose training was interrupted after epoch 30. We can load that checkpoint and continue from where we left off.

In [ ]:
# Create a fresh model and optimizer
resumed_model = MLP(CONFIG["input_dim"], CONFIG["hidden_dim"]).to(DEVICE)
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=CONFIG["lr"])

# Load checkpoint from epoch 30
ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "checkpoint_epoch_0030.pt")

if os.path.exists(ckpt_path):
    state = load_checkpoint(ckpt_path, resumed_model, resumed_optimizer, device=DEVICE)
    start_epoch = state["epoch"] + 1
    print(f"Resumed from epoch {state['epoch']} (val_loss was {state['loss']:.4f})")
    print(f"Will continue training from epoch {start_epoch}")
else:
    print(f"Checkpoint not found at {ckpt_path}. Run the training loop above first.")
    start_epoch = 1

In [ ]:
# Continue for 10 more epochs after the loaded checkpoint
EXTRA_EPOCHS = 10

for epoch in range(start_epoch, start_epoch + EXTRA_EPOCHS):
    resumed_model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        resumed_optimizer.zero_grad()
        preds = resumed_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        resumed_optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_loss = epoch_loss / N_TRAIN

    resumed_model.eval()
    with torch.no_grad():
        val_loss = criterion(resumed_model(X_val), y_val).item()

    print(f"Resumed epoch {epoch:3d} | train={train_loss:.4f} | val={val_loss:.4f}")

print("\nResume successful. Notice the loss picks up from where the original run left off.")

### Inspecting the Logged Metrics

In [ ]:
import csv

rows = run_csv.read()
print(f"Total logged epochs: {len(rows)}")
print("\nFirst 5 rows:")
for row in rows[:5]:
    print(f"  epoch={row['epoch']}  train={row['train_loss']}  val={row['val_loss']}")

print("\nLast 5 rows:")
for row in rows[-5:]:
    print(f"  epoch={row['epoch']}  train={row['train_loss']}  val={row['val_loss']}")

## Exercise

1. **Seed sensitivity**: Change `CONFIG["seed"]` from 42 to 0, 1, and 100. Run the training loop for each. Record the final validation loss. How much does it vary? Does the best-of-3 look meaningfully different from the worst-of-3?

2. **Checkpoint loading verification**: After the training loop finishes, load `best.pt` into a new model. Compute its validation loss. Confirm it matches the lowest val loss printed during training.

3. **Add learning rate to the CSV**: Modify `CSVLogger` setup to include an `lr` column. Then add a learning rate scheduler (`torch.optim.lr_scheduler.CosineAnnealingLR`) to the training loop and log `scheduler.get_last_lr()[0]` each epoch.

4. **(Stretch)** Add early stopping: if the validation loss does not improve for 10 consecutive epochs, stop training and print a message. Use a counter variable that resets to 0 whenever `is_best` is `True`.

## 5. Structured Hyperparameter Config with `dataclasses`

Scattered `args` variables or global constants at the top of a script are hard to track, hard to serialize, and easy to forget. A `dataclass` turns your config into a proper typed object that can be printed, passed around, serialized to JSON, and logged verbatim to W&B or MLflow.

Advantages over `argparse` namespaces or plain dicts:
- Type annotations serve as documentation.
- IDE autocomplete works on config fields.
- `dataclasses.asdict()` gives a clean dict for logging.
- Default values are explicit and centralized.

In [ ]:
from dataclasses import dataclass, asdict, field
import json

@dataclass
class TrainConfig:
    # Data
    input_dim: int = 20
    n_train: int = 2000
    n_val: int = 400

    # Model
    hidden_dim: int = 128
    n_layers: int = 2

    # Optimization
    lr: float = 1e-3
    batch_size: int = 64
    epochs: int = 50
    weight_decay: float = 1e-4

    # Reproducibility
    seed: int = 42

    # I/O
    checkpoint_dir: str = "runs/config_demo/checkpoints"
    metrics_csv: str = "runs/config_demo/metrics.csv"
    checkpoint_every: int = 10

    # Scheduler
    use_cosine_lr: bool = True
    warmup_epochs: int = 5

    def to_dict(self) -> dict:
        """Return a plain dict (useful for logging to W&B / MLflow)."""
        return asdict(self)

    def to_json(self, indent: int = 2) -> str:
        """Serialize to a JSON string for saving alongside checkpoints."""
        return json.dumps(self.to_dict(), indent=indent)

    @classmethod
    def from_dict(cls, d: dict) -> "TrainConfig":
        """Reconstruct a config from a dict (e.g., loaded from a checkpoint)."""
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


# Instantiate with defaults
cfg = TrainConfig()
print("Default config:")
print(cfg)
print()

# Override a few fields for a different experiment
fast_cfg = TrainConfig(epochs=5, lr=3e-3, seed=0)
print("Fast-run config (overrides):")
print(fast_cfg)
print()

# Serialize for logging
print("JSON representation (for saving alongside checkpoints):")
print(cfg.to_json())

## 6. Checkpointing with `torch.save` / `torch.load`

The checkpoint functions introduced earlier in this notebook save and load full training state. This section zooms in on the exact format of what gets written to disk and shows how to inspect a saved checkpoint without loading it into a model.

Key things to understand:
- `model.state_dict()` returns an `OrderedDict` of parameter tensors. It does not include the model architecture, so you need to instantiate the same model class before calling `load_state_dict`.
- `optimizer.state_dict()` contains momentum buffers and per-parameter learning rate state. Without this, resumed training restarts momentum from zero, which hurts convergence for the first few steps.
- Saving the config dict alongside the weights means you can always answer "what hyperparameters produced this checkpoint?"

In [ ]:
import torch
import torch.nn as nn
import os

# Build a small model and optimizer just to demonstrate the checkpoint format.
set_seed(42)
demo_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 1)
)
demo_optimizer = torch.optim.Adam(demo_model.parameters(), lr=1e-3)

# Simulate one gradient step so the optimizer has non-trivial state.
dummy_x = torch.randn(8, 20)
dummy_y = torch.randn(8)
loss = nn.MSELoss()(demo_model(dummy_x).squeeze(), dummy_y)
loss.backward()
demo_optimizer.step()
demo_optimizer.zero_grad()

# -----------------------------------------------------------------------
# Save a full checkpoint
# -----------------------------------------------------------------------
os.makedirs("runs/ckpt_demo", exist_ok=True)

checkpoint = {
    "epoch": 7,
    "loss": loss.item(),
    "model_state_dict": demo_model.state_dict(),
    "optimizer_state_dict": demo_optimizer.state_dict(),
    "config": TrainConfig().to_dict(),
}

ckpt_path = "runs/ckpt_demo/checkpoint_epoch_0007.pt"
torch.save(checkpoint, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")
print(f"File size: {os.path.getsize(ckpt_path) / 1024:.1f} KB")

# -----------------------------------------------------------------------
# Inspect a checkpoint without loading a model
# -----------------------------------------------------------------------
loaded = torch.load(ckpt_path, map_location="cpu")

print("\nCheckpoint keys:", list(loaded.keys()))
print(f"Saved at epoch : {loaded['epoch']}")
print(f"Loss at save   : {loaded['loss']:.6f}")
print(f"Config (seed)  : {loaded['config']['seed']}")
print(f"Config (lr)    : {loaded['config']['lr']}")

print("\nmodel_state_dict keys and shapes:")
for name, tensor in loaded["model_state_dict"].items():
    print(f"  {name:30s}  {list(tensor.shape)}")

print("\noptimizer_state_dict param_groups[0] keys:")
pg = loaded["optimizer_state_dict"]["param_groups"][0]
print("  " + ", ".join(pg.keys()))

# -----------------------------------------------------------------------
# Load back into a fresh model and verify weights match exactly
# -----------------------------------------------------------------------
restored_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 1)
)
restored_model.load_state_dict(loaded["model_state_dict"])

# All parameter tensors should be identical
all_match = all(
    torch.equal(p1, p2)
    for p1, p2 in zip(demo_model.parameters(), restored_model.parameters())
)
print(f"\nAll weights match after reload: {all_match}")

## 7. Full W&B Run: `init`, per-step logging, artifacts, `finish`

Weights & Biases stores your metrics in the cloud and lets you compare runs in a browser. The minimal API is four calls:

1. `wandb.init(project=..., config=...)` -- starts a run and records the hyperparameters.
2. `wandb.log({...})` -- called at each step or epoch, appends a row to the run's metric table.
3. `wandb.log_artifact(...)` -- uploads a file (model checkpoint, dataset, etc.) and versions it.
4. `wandb.finish()` -- closes the run. Important in notebooks, otherwise the run stays "running" indefinitely.

The cell below shows the full pattern against the MLP from the hands-on section. It guards every W&B call behind `WANDB_AVAILABLE` so the notebook still runs correctly if W&B is not installed.

In [ ]:
# pip install wandb  # uncomment if needed
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import os

WANDB_AVAILABLE = False
try:
    import wandb
    WANDB_AVAILABLE = True
    print("wandb is installed. Full W&B logging enabled.")
except ImportError:
    print("wandb not installed. Run `pip install wandb` and `wandb login` to enable cloud logging.")
    print("The training loop below will still run; W&B calls are skipped gracefully.")


# -----------------------------------------------------------------------
# Setup: reuse the TrainConfig dataclass from section 5
# -----------------------------------------------------------------------
wb_cfg = TrainConfig(epochs=30, lr=1e-3, seed=42, hidden_dim=64)
set_seed(wb_cfg.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Synthetic data (same setup as the hands-on section)
X_all = torch.randn(wb_cfg.n_train + wb_cfg.n_val, wb_cfg.input_dim)
true_w = torch.zeros(wb_cfg.input_dim)
true_w[:5] = torch.tensor([1.0, -2.0, 0.5, 1.5, -1.0])
y_all = X_all @ true_w + 0.2 * torch.randn(wb_cfg.n_train + wb_cfg.n_val)

X_train = X_all[:wb_cfg.n_train].to(DEVICE)
y_train = y_all[:wb_cfg.n_train].to(DEVICE)
X_val   = X_all[wb_cfg.n_train:].to(DEVICE)
y_val   = y_all[wb_cfg.n_train:].to(DEVICE)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=wb_cfg.batch_size,
    shuffle=True
)

wb_model = MLP(wb_cfg.input_dim, wb_cfg.hidden_dim).to(DEVICE)
wb_optimizer = torch.optim.Adam(wb_model.parameters(), lr=wb_cfg.lr)
criterion = nn.MSELoss()

# -----------------------------------------------------------------------
# W&B init -- call once at the start
# -----------------------------------------------------------------------
run = None
if WANDB_AVAILABLE:
    run = wandb.init(
        project="til-ai-hygiene-demo",
        name="mlp-synthetic-v1",
        config=wb_cfg.to_dict(),   # logs all hyperparameters
        tags=["mlp", "synthetic", "demo"],
    )
    print(f"W&B run URL: {run.url}")

# -----------------------------------------------------------------------
# Training loop with per-step logging
# -----------------------------------------------------------------------
global_step = 0
best_val_loss = float("inf")
os.makedirs("runs/wb_demo/checkpoints", exist_ok=True)

for epoch in range(1, wb_cfg.epochs + 1):
    wb_model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        wb_optimizer.zero_grad()
        preds = wb_model(xb)
        loss = criterion(preds.squeeze(), yb)
        loss.backward()
        wb_optimizer.step()
        epoch_loss += loss.item() * len(xb)
        global_step += 1

        # Log every 10 steps to W&B
        if WANDB_AVAILABLE and global_step % 10 == 0:
            wandb.log({"train/loss_step": loss.item(), "step": global_step})

    train_loss = epoch_loss / wb_cfg.n_train

    wb_model.eval()
    with torch.no_grad():
        val_loss = criterion(wb_model(X_val).squeeze(), y_val).item()

    # Log per-epoch metrics
    if WANDB_AVAILABLE:
        wandb.log({
            "train/loss_epoch": train_loss,
            "val/loss": val_loss,
            "epoch": epoch,
            "lr": wb_optimizer.param_groups[0]["lr"],
        })

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        best_ckpt_path = "runs/wb_demo/checkpoints/best.pt"
        torch.save({
            "epoch": epoch,
            "loss": val_loss,
            "model_state_dict": wb_model.state_dict(),
            "optimizer_state_dict": wb_optimizer.state_dict(),
            "config": wb_cfg.to_dict(),
        }, best_ckpt_path)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{wb_cfg.epochs}  train={train_loss:.4f}  val={val_loss:.4f}"
              + ("  *" if is_best else ""))

# -----------------------------------------------------------------------
# Log the best checkpoint as a W&B artifact
# -----------------------------------------------------------------------
if WANDB_AVAILABLE and os.path.exists(best_ckpt_path):
    artifact = wandb.Artifact(
        name="mlp-best-checkpoint",
        type="model",
        description=f"Best MLP checkpoint (val_loss={best_val_loss:.4f})",
        metadata=wb_cfg.to_dict(),
    )
    artifact.add_file(best_ckpt_path)
    run.log_artifact(artifact)
    print(f"Logged artifact: mlp-best-checkpoint")

# -----------------------------------------------------------------------
# finish() closes the run. Always call this in notebooks.
# -----------------------------------------------------------------------
if WANDB_AVAILABLE:
    wandb.finish()
    print("W&B run finished.")

print(f"\nTraining done. Best val loss: {best_val_loss:.4f}")

## 8. MLflow: An Alternative Experiment Tracker

MLflow is an open-source alternative to W&B that runs entirely locally (no account needed) and stores runs in a directory on disk. The UI starts with `mlflow ui` in the terminal and opens at `http://localhost:5000`.

The API mirrors W&B in spirit:

| W&B | MLflow |
|---|---|
| `wandb.init(config=...)` | `mlflow.start_run()` + `mlflow.log_params(...)` |
| `wandb.log({"loss": v})` | `mlflow.log_metric("loss", v, step=s)` |
| `wandb.log_artifact(path)` | `mlflow.log_artifact(path)` |
| `wandb.finish()` | `mlflow.end_run()` (or use context manager) |

MLflow is a good choice when:
- You want everything on your own machine with no cloud dependency.
- Your organization has data-residency requirements.
- You are already using the MLflow model registry for deployment.

In [ ]:
# pip install mlflow  # uncomment if needed

MLFLOW_AVAILABLE = False
try:
    import mlflow
    import mlflow.pytorch
    MLFLOW_AVAILABLE = True
    print("mlflow is installed.")
except ImportError:
    print("mlflow not installed. Run `pip install mlflow` to enable.")
    print("The training code below will still run; MLflow calls are skipped.")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import os

# -----------------------------------------------------------------------
# Reuse the same model class and data from earlier in the notebook.
# -----------------------------------------------------------------------
mlf_cfg = TrainConfig(epochs=20, lr=1e-3, seed=7, hidden_dim=64)
set_seed(mlf_cfg.seed)

mlf_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_mlf = torch.randn(mlf_cfg.n_train + mlf_cfg.n_val, mlf_cfg.input_dim)
true_w = torch.zeros(mlf_cfg.input_dim)
true_w[:5] = torch.tensor([1.0, -2.0, 0.5, 1.5, -1.0])
y_mlf = X_mlf @ true_w + 0.2 * torch.randn(mlf_cfg.n_train + mlf_cfg.n_val)

X_tr = X_mlf[:mlf_cfg.n_train].to(mlf_device)
y_tr = y_mlf[:mlf_cfg.n_train].to(mlf_device)
X_vl = X_mlf[mlf_cfg.n_train:].to(mlf_device)
y_vl = y_mlf[mlf_cfg.n_train:].to(mlf_device)

mlf_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=mlf_cfg.batch_size, shuffle=True)
mlf_model = MLP(mlf_cfg.input_dim, mlf_cfg.hidden_dim).to(mlf_device)
mlf_optimizer = torch.optim.Adam(mlf_model.parameters(), lr=mlf_cfg.lr)
mlf_criterion = nn.MSELoss()

os.makedirs("runs/mlflow_demo/checkpoints", exist_ok=True)

# -----------------------------------------------------------------------
# MLflow run
# -----------------------------------------------------------------------
if MLFLOW_AVAILABLE:
    # Store runs in a local directory (creates mlruns/ by default)
    mlflow.set_tracking_uri("runs/mlflow_demo/mlruns")
    mlflow.set_experiment("til-ai-hygiene-demo")

# Use a context manager so the run closes even if training throws an error.
def run_with_mlflow():
    if MLFLOW_AVAILABLE:
        ctx = mlflow.start_run(run_name="mlp-synthetic-mlflow")
    else:
        from contextlib import nullcontext
        ctx = nullcontext()

    with ctx:
        # --- Log hyperparameters ---
        if MLFLOW_AVAILABLE:
            # log_params expects string values; MLflow converts automatically for dicts.
            mlflow.log_params(mlf_cfg.to_dict())

        best_val_loss = float("inf")
        step = 0

        for epoch in range(1, mlf_cfg.epochs + 1):
            mlf_model.train()
            epoch_loss = 0.0
            for xb, yb in mlf_loader:
                mlf_optimizer.zero_grad()
                loss = mlf_criterion(mlf_model(xb).squeeze(), yb)
                loss.backward()
                mlf_optimizer.step()
                epoch_loss += loss.item() * len(xb)
                step += 1

                # --- Log step-level metric ---
                if MLFLOW_AVAILABLE and step % 10 == 0:
                    mlflow.log_metric("train_loss_step", loss.item(), step=step)

            train_loss = epoch_loss / mlf_cfg.n_train

            mlf_model.eval()
            with torch.no_grad():
                val_loss = mlf_criterion(mlf_model(X_vl).squeeze(), y_vl).item()

            # --- Log epoch-level metrics ---
            if MLFLOW_AVAILABLE:
                mlflow.log_metrics({
                    "train_loss_epoch": train_loss,
                    "val_loss": val_loss,
                }, step=epoch)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                ckpt_path = "runs/mlflow_demo/checkpoints/best.pt"
                torch.save({
                    "epoch": epoch,
                    "loss": val_loss,
                    "model_state_dict": mlf_model.state_dict(),
                    "optimizer_state_dict": mlf_optimizer.state_dict(),
                    "config": mlf_cfg.to_dict(),
                }, ckpt_path)

            if epoch % 5 == 0:
                print(f"Epoch {epoch:3d}/{mlf_cfg.epochs}  train={train_loss:.4f}  val={val_loss:.4f}")

        # --- Log the best checkpoint as an artifact ---
        if MLFLOW_AVAILABLE and os.path.exists(ckpt_path):
            mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
            print(f"Logged artifact: {ckpt_path}")

            # Optionally log the model in MLflow's native format for the registry:
            # mlflow.pytorch.log_model(mlf_model, artifact_path="model")

        print(f"\nBest val loss: {best_val_loss:.4f}")

        if MLFLOW_AVAILABLE:
            run_id = mlflow.active_run().info.run_id
            print(f"MLflow run ID: {run_id}")
            print("View runs with: mlflow ui --backend-store-uri runs/mlflow_demo/mlruns")

run_with_mlflow()

## 9. Tracing Code Versions with Git Commit Hashes

An experiment log without a code version is unreliable. Six months later you won't remember which version of your model file produced which set of numbers. The fix is to record the git commit hash alongside every run's metadata.

Two lines of Python are enough:

```python
import subprocess
commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
```

You can then embed this hash in:
- Your checkpoint's `config` dict
- Every W&B / MLflow run's parameters
- Your CSV log's header row

The cell below shows a helper that collects the full set of useful git metadata for an experiment record.

In [ ]:
import subprocess
import datetime

def get_git_metadata() -> dict:
    """
    Collect git metadata for embedding in experiment records.

    Returns a dict with:
      - commit_hash: full SHA of HEAD
      - commit_short: first 8 chars (human-friendly)
      - branch: current branch name
      - commit_message: subject line of the HEAD commit
      - author: name and email of the commit author
      - is_dirty: True if there are uncommitted changes (results may not be
                  reproducible from the recorded hash alone)
    """
    def _git(*args) -> str:
        try:
            return subprocess.check_output(
                ["git"] + list(args),
                stderr=subprocess.DEVNULL
            ).decode().strip()
        except (subprocess.CalledProcessError, FileNotFoundError):
            return "unknown"

    commit_hash = _git("rev-parse", "HEAD")
    branch = _git("rev-parse", "--abbrev-ref", "HEAD")
    commit_message = _git("log", "-1", "--format=%s")
    author = _git("log", "-1", "--format=%an <%ae>")

    # Check for uncommitted changes (exit code 1 = dirty tree)
    try:
        subprocess.check_call(
            ["git", "diff-index", "--quiet", "HEAD", "--"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        is_dirty = False
    except (subprocess.CalledProcessError, FileNotFoundError):
        is_dirty = True

    return {
        "commit_hash": commit_hash,
        "commit_short": commit_hash[:8] if commit_hash != "unknown" else "unknown",
        "branch": branch,
        "commit_message": commit_message,
        "author": author,
        "is_dirty": is_dirty,
        "recorded_at": datetime.datetime.utcnow().isoformat() + "Z",
    }


git_meta = get_git_metadata()
print("Git metadata for this experiment:")
for k, v in git_meta.items():
    print(f"  {k:20s}: {v}")

if git_meta["is_dirty"]:
    print("\nWARNING: working tree has uncommitted changes.")
    print("Results logged under this commit hash may not be exactly reproducible.")
    print("Commit your changes before running production experiments.")

In [ ]:
# Demonstrate embedding git metadata in a checkpoint and a W&B / MLflow run.

import torch, os

# Build the combined experiment record: config + git provenance
experiment_record = {
    **TrainConfig().to_dict(),
    "git": get_git_metadata(),
}

print("Full experiment record (what you would pass to wandb.init(config=...) or mlflow.log_params(...)):")
print(json.dumps(experiment_record, indent=2))

# Save alongside a checkpoint so any future reader can trace this file back to a commit.
os.makedirs("runs/git_demo", exist_ok=True)
with open("runs/git_demo/experiment_record.json", "w") as f:
    json.dump(experiment_record, f, indent=2)

print("\nSaved to runs/git_demo/experiment_record.json")
print(f"\nTo recover the exact code: git checkout {git_meta['commit_short']}")

## Exercise

Write a training loop from scratch that combines everything in this notebook.

**Requirements:**

1. Use `TrainConfig` to hold all hyperparameters.
2. Call `set_seed` before any data or model initialization.
3. Train an `MLP` on the synthetic regression dataset for the number of epochs in the config.
4. Log `train/loss` and `val/loss` to W&B every 10 steps (use the `WANDB_AVAILABLE` guard so it still runs without W&B).
5. Save a checkpoint at the end of every epoch. The checkpoint must include `epoch`, `model_state_dict`, `optimizer_state_dict`, `loss`, and `config`.
6. Implement resume-from-checkpoint: before the training loop starts, check if a checkpoint file exists at `cfg.checkpoint_dir + "/latest.pt"`. If it does, load it and start the loop from `saved_epoch + 1` rather than epoch 1.
7. Embed `get_git_metadata()` in the config passed to `wandb.init`.

The skeleton below has `# YOUR CODE HERE` markers. Fill them in. Do not remove the `raise NotImplementedError` lines until you have implemented the corresponding section.

In [ ]:
# pip install torch wandb  # uncomment if needed
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import os

# -----------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------
cfg = TrainConfig(
    epochs=40,
    lr=1e-3,
    seed=42,
    hidden_dim=64,
    checkpoint_dir="runs/exercise/checkpoints",
    checkpoint_every=1,  # save every epoch for the resume test
)

# -----------------------------------------------------------------------
# 2. Seed everything
# -----------------------------------------------------------------------
# YOUR CODE HERE: call set_seed with cfg.seed
raise NotImplementedError("Call set_seed here")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------
# 3. Data
# -----------------------------------------------------------------------
# YOUR CODE HERE: create X_train, y_train, X_val, y_val and a DataLoader
# Use the same synthetic regression setup from the hands-on section:
#   - cfg.input_dim features, cfg.n_train + cfg.n_val samples
#   - true weights: 1.0, -2.0, 0.5, 1.5, -1.0 on first 5 dims, 0 elsewhere
#   - add Gaussian noise with std=0.2
raise NotImplementedError("Create dataset and DataLoader here")

# -----------------------------------------------------------------------
# 4. Model and optimizer
# -----------------------------------------------------------------------
# YOUR CODE HERE: instantiate MLP and Adam optimizer
model = None
optimizer = None
raise NotImplementedError("Instantiate model and optimizer here")

criterion = nn.MSELoss()

# -----------------------------------------------------------------------
# 5. W&B init (guarded)
# -----------------------------------------------------------------------
# YOUR CODE HERE: call wandb.init with project="til-ai-exercise",
# config = cfg.to_dict() merged with {"git": get_git_metadata()}
# Use the WANDB_AVAILABLE guard.
run = None
raise NotImplementedError("Initialize W&B run here (with WANDB_AVAILABLE guard)")

# -----------------------------------------------------------------------
# 6. Resume from checkpoint
# -----------------------------------------------------------------------
latest_ckpt = os.path.join(cfg.checkpoint_dir, "latest.pt")
start_epoch = 1

# YOUR CODE HERE: if latest_ckpt exists, load it and set start_epoch correctly
raise NotImplementedError("Implement checkpoint resume logic here")

# -----------------------------------------------------------------------
# 7. Training loop
# -----------------------------------------------------------------------
os.makedirs(cfg.checkpoint_dir, exist_ok=True)
global_step = 0
best_val_loss = float("inf")

for epoch in range(start_epoch, cfg.epochs + 1):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        # YOUR CODE HERE: forward pass, loss, backward, optimizer step
        raise NotImplementedError("Implement the training step")

        epoch_loss += loss.item() * len(xb)
        global_step += 1

        # YOUR CODE HERE: log train/loss_step to wandb every 10 steps
        raise NotImplementedError("Add per-step W&B logging here")

    train_loss = epoch_loss / cfg.n_train

    model.eval()
    with torch.no_grad():
        # YOUR CODE HERE: compute val_loss
        val_loss = None
        raise NotImplementedError("Compute validation loss here")

    # YOUR CODE HERE: log train_loss and val_loss to wandb per epoch
    raise NotImplementedError("Add per-epoch W&B logging here")

    # YOUR CODE HERE: save checkpoint to latest_ckpt every epoch
    # The checkpoint dict must contain: epoch, model_state_dict,
    # optimizer_state_dict, loss (val_loss), config
    raise NotImplementedError("Save checkpoint here")

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{cfg.epochs}  train={train_loss:.4f}  val={val_loss:.4f}")

# -----------------------------------------------------------------------
# 8. Finish W&B run
# -----------------------------------------------------------------------
# YOUR CODE HERE: call wandb.finish() if WANDB_AVAILABLE
raise NotImplementedError("Finish the W&B run here")

print(f"Training complete. Best val loss: {best_val_loss:.4f}")